# Systematic Trading Case Study

## Cross-Sectional Momentum with Volatility Scaling and Transaction Costs

You are a quantitative researcher at a systematic asset manager.

Your task is to research and backtest a **cross-sectional momentum strategy** on a synthetic universe of liquid futures-like assets.

The notebook intentionally provides:

- realistic but synthetic market data;
- explicit research objectives;
- incomplete code cells for you to fill;
- validation checks;
- progressively harder exercises;
- no final implementation.

The focus is on:

- pandas and NumPy fluency;
- signal construction;
- avoiding look-ahead bias;
- portfolio normalization;
- volatility targeting;
- turnover and transaction costs;
- performance attribution;
- robustness analysis.

## Rules

1. Do not use explicit Python loops unless the exercise permits them.
2. Prefer vectorized pandas/NumPy operations.
3. All positions used for day `t` returns must be based only on information available no later than day `t-1`.
4. Preserve indices and column labels.
5. Read every exercise carefully before coding.


## 0. Setup and synthetic market data

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

TRADING_DAYS = 252
rng = np.random.default_rng(42)

dates = pd.bdate_range("2012-01-02", "2025-12-31")

assets = [
    "ES",   # US equities
    "NQ",   # US technology equities
    "DAX",  # European equities
    "NKY",  # Japanese equities
    "TY",   # US 10Y bonds
    "BUND", # German bonds
    "GOLD",
    "OIL",
    "COPPER",
    "EURUSD",
    "JPYUSD",
    "GBPUSD",
]

asset_class = pd.Series({
    "ES": "Equity",
    "NQ": "Equity",
    "DAX": "Equity",
    "NKY": "Equity",
    "TY": "Rates",
    "BUND": "Rates",
    "GOLD": "Commodity",
    "OIL": "Commodity",
    "COPPER": "Commodity",
    "EURUSD": "FX",
    "JPYUSD": "FX",
    "GBPUSD": "FX",
}, name="asset_class")

n_dates = len(dates)
n_assets = len(assets)

# Four latent factors: global risk, rates, commodities, USD.
factor_vols = np.array([0.0080, 0.0045, 0.0090, 0.0055])
factor_corr = np.array([
    [1.00, -0.20,  0.35, -0.15],
    [-0.20, 1.00, -0.10,  0.10],
    [0.35, -0.10, 1.00, -0.05],
    [-0.15, 0.10, -0.05, 1.00],
])
factor_cov = np.outer(factor_vols, factor_vols) * factor_corr

factors = rng.multivariate_normal(
    mean=np.zeros(4),
    cov=factor_cov,
    size=n_dates
)

loadings = pd.DataFrame(
    {
        "global_risk": [1.0, 1.2, 0.9, 0.8, -0.25, -0.20, 0.10, 0.25, 0.45, -0.10, -0.05, -0.08],
        "rates":       [-0.1, -0.1, -0.1, -0.1,  1.10,  1.00, 0.15, -0.10, -0.10, 0.10, 0.20, 0.10],
        "commodity":   [0.1, 0.1, 0.2, 0.2, -0.10, -0.10, 0.80, 1.10, 0.95, 0.00, 0.00, 0.00],
        "usd":         [0.0, 0.0, 0.0, 0.0,  0.00,  0.00, -0.20, -0.20, -0.15, -1.00, 0.75, -0.85],
    },
    index=assets,
)

idio_vol = pd.Series({
    "ES": 0.0060,
    "NQ": 0.0080,
    "DAX": 0.0070,
    "NKY": 0.0075,
    "TY": 0.0030,
    "BUND": 0.0030,
    "GOLD": 0.0060,
    "OIL": 0.0120,
    "COPPER": 0.0090,
    "EURUSD": 0.0045,
    "JPYUSD": 0.0050,
    "GBPUSD": 0.0048,
})

# Persistent trend component so momentum has an economically meaningful signal.
trend = np.zeros((n_dates, n_assets))
trend_shocks = rng.normal(0, 0.00045, size=(n_dates, n_assets))
for t in range(1, n_dates):
    trend[t] = 0.985 * trend[t - 1] + trend_shocks[t]

common_returns = factors @ loadings.to_numpy().T
idio_returns = rng.normal(0, idio_vol.to_numpy(), size=(n_dates, n_assets))

returns = pd.DataFrame(
    common_returns + idio_returns + trend,
    index=dates,
    columns=assets,
)

# Add a few crisis-like shocks.
shock_dates = {
    "2015-08-24": -0.040,
    "2020-03-16": -0.085,
    "2022-06-13": -0.035,
}
for date, shock in shock_dates.items():
    if date in returns.index:
        returns.loc[date, ["ES", "NQ", "DAX", "NKY"]] += shock
        returns.loc[date, ["TY", "BUND", "GOLD"]] -= 0.25 * shock

prices = 100 * np.exp(returns.cumsum())

# Synthetic daily dollar volume used later for liquidity filters and costs.
base_adv = pd.Series({
    "ES": 25e9,
    "NQ": 18e9,
    "DAX": 8e9,
    "NKY": 7e9,
    "TY": 15e9,
    "BUND": 10e9,
    "GOLD": 6e9,
    "OIL": 9e9,
    "COPPER": 3e9,
    "EURUSD": 30e9,
    "JPYUSD": 22e9,
    "GBPUSD": 14e9,
})

volume_noise = rng.lognormal(mean=0.0, sigma=0.35, size=(n_dates, n_assets))
dollar_volume = pd.DataFrame(
    volume_noise * base_adv.to_numpy(),
    index=dates,
    columns=assets,
)

returns.head()

,ES,NQ,DAX,NKY,TY,BUND,GOLD,OIL,COPPER,EURUSD,JPYUSD,GBPUSD
2012-01-02,0.0103,-0.0054,-0.0045,-0.0088,-0.0062,-0.0074,-0.0069,-0.0056,-0.0164,0.0101,0.0013,0.0011
2012-01-03,0.0284,0.0267,0.0261,0.0077,-0.0094,-0.0053,0.0105,0.0151,0.0180,0.0044,-0.0039,0.0099
2012-01-04,-0.0033,0.0052,0.0074,-0.0056,-0.0065,-0.0049,0.0075,0.0015,0.0062,0.0087,-0.0188,0.0042
2012-01-05,-0.0091,0.0052,-0.0002,0.0031,0.0076,0.0048,0.0088,0.0036,0.0130,-0.0024,0.0048,0.0007
2012-01-06,0.0053,-0.0103,0.0022,-0.0003,0.0000,-0.0066,-0.0045,-0.0030,-0.0195,0.0006,-0.0050,0.0067


## 1. Data audit

Before researching a strategy, verify the dataset.

### Tasks

Create the following objects:

- `audit`: a DataFrame indexed by asset containing:
  - start date;
  - end date;
  - number of observations;
  - number of missing observations;
  - annualized mean return;
  - annualized volatility;
  - minimum daily return;
  - maximum daily return.
- `corr`: the full-sample return correlation matrix.

Then identify the three most highly correlated **distinct** asset pairs.

Store them in `top_corr_pairs` as a Series with a two-level index.

### Constraints

- Do not manually enumerate pairs.
- Do not include diagonal correlations.
- Do not include both `(A, B)` and `(B, A)`.


In [2]:
# TODO: build audit, corr, and top_corr_pairs
audit = pd.DataFrame({'start': min(returns.index)}, index=returns.columns)
audit['end'] = max(returns.index)
audit['n_obs'] = len(returns)
audit['n_missing'] = returns.isna().sum().to_numpy()
audit['ann_mean'] = returns.mean().mul(TRADING_DAYS)
audit['ann_vol'] = returns.std().mul(np.sqrt(TRADING_DAYS))
audit['min_return'] = returns.min()
audit['max_return'] = returns.max()
display(audit.head(5))

corr = returns.corr()
upper_triangle = corr.where(
    np.triu(np.ones(corr.shape), k=1).astype(bool) # k=1 to set the diagonal as NaN, 0 would not mask it but only below
)
display(upper_triangle)
corr_pairs = upper_triangle.stack()
top_corr_pairs = pd.Series(corr_pairs.nlargest(3))
top_corr_pairs


,start,end,n_obs,n_missing,ann_mean,ann_vol,min_return,max_return
ES,2012-01-02,2025-12-31,3653,0,0.1655,0.1733,-0.0934,0.0355
NQ,2012-01-02,2025-12-31,3653,0,-0.0997,0.2072,-0.0780,0.0422
DAX,2012-01-02,2025-12-31,3653,0,-0.1737,0.1781,-0.1093,0.0345
NKY,2012-01-02,2025-12-31,3653,0,0.0370,0.1735,-0.1044,0.0399
TY,2012-01-02,2025-12-31,3653,0,0.0149,0.1158,-0.0252,0.0257


,ES,NQ,DAX,NKY,TY,BUND,GOLD,OIL,COPPER,EURUSD,JPYUSD,GBPUSD
ES,NaN,0.6239,0.5859,0.5434,-0.4051,-0.3543,0.2614,0.2884,0.4277,0.0148,-0.1556,0.0167
NQ,NaN,NaN,0.5807,0.5034,-0.3774,-0.3158,0.2666,0.2781,0.4044,0.0085,-0.1106,0.0214
DAX,NaN,NaN,NaN,0.5179,-0.3766,-0.3056,0.3006,0.3074,0.4319,0.0059,-0.1079,0.0217
NKY,NaN,NaN,NaN,NaN,-0.3495,-0.3224,0.3034,0.3095,0.4033,0.0081,-0.1353,-0.0063
TY,NaN,NaN,NaN,NaN,NaN,0.6596,-0.1564,-0.2386,-0.3214,0.0011,0.2229,0.0196
BUND,NaN,NaN,NaN,NaN,NaN,NaN,-0.1329,-0.1967,-0.2771,0.0171,0.2025,0.0611
GOLD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.4659,0.5045,0.0886,-0.0951,0.0809
OIL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.4905,0.0721,-0.0763,0.0522
COPPER,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0558,-0.1183,0.0499
EURUSD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.3832,0.4668


TY  BUND   0.6596
ES  NQ     0.6239
    DAX    0.5859
dtype: float64

In [3]:
# Validation
assert audit.index.equals(pd.Index(assets))
assert {"start", "end", "n_obs", "n_missing", "ann_mean", "ann_vol", "min_return", "max_return"} <= set(audit.columns)
assert corr.shape == (n_assets, n_assets)
assert len(top_corr_pairs) == 3
assert isinstance(top_corr_pairs.index, pd.MultiIndex)
print("Exercise 1 passed.")

Exercise 1 passed.


## 2. Volatility estimation

Estimate each asset's ex-ante volatility using a 60-day rolling standard deviation.

### Tasks

Create:

- `daily_vol_60`: rolling daily volatility;
- `ann_vol_60`: annualized rolling volatility;
- `inv_vol`: inverse-volatility scores.

### Important

Shift the volatility estimate by one day before it can be used for trading.

At date `t`, your estimate must only depend on returns observed through `t-1`.

Clip annualized volatility to a minimum of 5% to prevent extreme leverage.


In [ ]:
# TODO: create daily_vol_60, ann_vol_60, and inv_vol



In [ ]:
# Validation
assert daily_vol_60.shape == returns.shape
assert ann_vol_60.shape == returns.shape
assert inv_vol.shape == returns.shape
assert ann_vol_60.min().min() >= 0.05
assert ann_vol_60.iloc[:59].isna().all().all()
print("Exercise 2 passed.")

## 3. Time-series momentum signal

Construct a classic 12-month momentum signal with a one-month skip.

For each asset and date:

\[
\text{momentum}_{t}
=
\frac{P_{t-21}}{P_{t-252}} - 1
\]

### Tasks

Create:

- `mom_12_1`: the raw 12-1 momentum return;
- `ts_signal`: the directional signal equal to `+1`, `0`, or `-1`.

### Important

The signal used on date `t` must be shifted so that today's return does not influence today's position.


In [ ]:
# TODO: create mom_12_1 and ts_signal



In [ ]:
# Validation
assert mom_12_1.shape == prices.shape
assert ts_signal.shape == prices.shape
assert set(np.unique(ts_signal.dropna().to_numpy())) <= {-1.0, 0.0, 1.0}
assert ts_signal.iloc[:253].isna().all().all()
print("Exercise 3 passed.")

## 4. Cross-sectional momentum score

Instead of using only the sign of each asset's own trend, rank assets against one another.

### Tasks

For every date:

1. Rank `mom_12_1` cross-sectionally.
2. Convert ranks to percentile ranks.
3. Map percentiles approximately into the interval `[-1, 1]`.
4. Demean the signal cross-sectionally so that the average score is zero.

Store the result in `cs_signal`.

### Interpretation

- Strong relative winners should have positive scores.
- Strong relative losers should have negative scores.
- The portfolio should be approximately dollar-neutral before volatility scaling.


In [ ]:
# TODO: create cs_signal



In [ ]:
# Validation
assert cs_signal.shape == returns.shape
row_means = cs_signal.mean(axis=1, skipna=True).dropna()
assert np.allclose(row_means, 0.0, atol=1e-12)
assert cs_signal.max().max() <= 1.0 + 1e-12
assert cs_signal.min().min() >= -1.0 - 1e-12
print("Exercise 4 passed.")

## 5. Raw portfolio construction

Construct a volatility-scaled cross-sectional momentum portfolio.

### Tasks

1. Multiply `cs_signal` by `inv_vol`.
2. Normalize each row so that gross exposure equals 1:
   \[
   \sum_i |w_{i,t}| = 1
   \]
3. Store the result in `weights_raw`.

### Edge case

Rows where all signals are missing or zero must remain missing or zero, not infinite.


In [ ]:
# TODO: create weights_raw



In [ ]:
# Validation
assert weights_raw.shape == returns.shape
gross = weights_raw.abs().sum(axis=1)
valid = gross[gross > 0]
assert np.allclose(valid, 1.0, atol=1e-10)
assert np.isfinite(weights_raw.fillna(0).to_numpy()).all()
print("Exercise 5 passed.")

## 6. Asset-class neutrality

The CIO does not want the strategy's performance to be dominated by persistent exposure to Equity, Rates, Commodity, or FX.

### Tasks

Create `weights_neutral` by:

1. Taking `weights_raw`.
2. Demeaning weights within each asset class on every date.
3. Renormalizing total gross exposure to 1.

### Requirement

For every valid date and each asset class, net exposure should be approximately zero.


In [ ]:
# TODO: create weights_neutral



In [ ]:
# Validation
assert weights_neutral.shape == returns.shape
for group in asset_class.unique():
    cols = asset_class[asset_class == group].index
    class_net = weights_neutral[cols].sum(axis=1).dropna()
    assert np.allclose(class_net, 0.0, atol=1e-10)

gross = weights_neutral.abs().sum(axis=1)
valid = gross[gross > 0]
assert np.allclose(valid, 1.0, atol=1e-10)
print("Exercise 6 passed.")

## 7. Backtest before transaction costs

Compute the portfolio return using lagged weights.

### Tasks

Create:

- `gross_strategy_returns`: daily strategy returns before costs;
- `gross_equity_curve`: cumulative growth of 1 monetary unit.

### Important

The return on date `t` must use positions decided at the end of date `t-1`.


In [ ]:
# TODO: create gross_strategy_returns and gross_equity_curve



In [ ]:
# Validation
assert isinstance(gross_strategy_returns, pd.Series)
assert gross_strategy_returns.index.equals(returns.index)
assert isinstance(gross_equity_curve, pd.Series)
assert gross_equity_curve.dropna().iloc[0] > 0
print("Exercise 7 passed.")

## 8. Turnover and linear transaction costs

Assume one-way trading cost depends on asset class:

| Asset class | Cost |
|---|---:|
| Equity | 1.5 bps |
| Rates | 1.0 bps |
| Commodity | 2.5 bps |
| FX | 0.8 bps |

### Tasks

Create:

- `cost_bps`: Series indexed by asset;
- `daily_turnover_by_asset`: absolute daily change in portfolio weights;
- `daily_turnover`: total daily turnover;
- `transaction_cost`: daily cost in return units;
- `net_strategy_returns`;
- `net_equity_curve`.

Use the convention:

\[
\text{cost}_t = \sum_i |w_{i,t} - w_{i,t-1}| 	imes \text{cost rate}_i
\]


In [ ]:
# TODO: create the transaction-cost objects



In [ ]:
# Validation
assert cost_bps.index.equals(pd.Index(assets))
assert (transaction_cost.dropna() >= 0).all()
assert (daily_turnover.dropna() >= 0).all()
common = pd.concat([gross_strategy_returns, net_strategy_returns], axis=1).dropna()
assert (common.iloc[:, 1] <= common.iloc[:, 0] + 1e-14).all()
print("Exercise 8 passed.")

## 9. Performance statistics

Write a function:

```python
performance_stats(r, periods_per_year=252)
```

It must return a Series with:

- annualized return;
- annualized volatility;
- Sharpe ratio, assuming zero risk-free rate;
- downside volatility;
- Sortino ratio;
- maximum drawdown;
- Calmar ratio;
- hit rate;
- skewness;
- excess kurtosis;
- worst daily return;
- best daily return.

Then create `stats` comparing gross and net strategy returns.

### Requirements

- Drop missing values inside the function.
- Use geometric annualized return.
- Maximum drawdown must be reported as a negative number.


In [ ]:
# TODO: define performance_stats and create stats



In [ ]:
# Validation
required_stats = {
    "ann_return", "ann_vol", "sharpe", "downside_vol", "sortino",
    "max_drawdown", "calmar", "hit_rate", "skew", "excess_kurtosis",
    "worst_day", "best_day"
}
assert required_stats <= set(stats.index)
assert {"Gross", "Net"} <= set(stats.columns)
assert stats.loc["max_drawdown"].max() <= 0
print("Exercise 9 passed.")

## 10. Volatility targeting

The portfolio currently has variable realized risk.

Target 10% annualized volatility using a 60-day rolling estimate of the strategy's own volatility.

### Tasks

Create:

- `strategy_ann_vol_60`;
- `leverage`;
- `targeted_returns`;
- `targeted_equity_curve`.

### Constraints

- Shift the volatility estimate by one day.
- Cap leverage at 2.0.
- Do not allow negative leverage.
- Apply leverage to **net** strategy returns.


In [ ]:
# TODO: create volatility-targeted strategy objects



In [ ]:
# Validation
assert leverage.max() <= 2.0 + 1e-12
assert leverage.min() >= 0.0
assert targeted_returns.index.equals(net_strategy_returns.index)
print("Exercise 10 passed.")

## 11. Drawdown analysis

Create a function:

```python
drawdown_table(r, top_n=5)
```

It must identify the largest drawdown episodes and return a DataFrame with:

- peak date;
- trough date;
- recovery date;
- drawdown depth;
- peak-to-trough duration in calendar days;
- recovery duration in calendar days.

If a drawdown has not recovered by the end of the sample, set the recovery date to `NaT`.

Apply it to `targeted_returns` and store the result in `dd_table`.


In [ ]:
# TODO: define drawdown_table and create dd_table



In [ ]:
# Validation
assert isinstance(dd_table, pd.DataFrame)
assert len(dd_table) <= 5
assert {"peak_date", "trough_date", "recovery_date", "drawdown",
        "peak_to_trough_days", "recovery_days"} <= set(dd_table.columns)
assert (dd_table["drawdown"] <= 0).all()
print("Exercise 11 passed.")

## 12. Signal efficacy: information coefficient

Test whether the signal predicts future returns.

### Tasks

1. Compute next-day returns for each asset.
2. For each date, calculate the cross-sectional Spearman correlation between:
   - today's `cs_signal`;
   - next day's asset returns.
3. Store the daily IC in `daily_ic`.
4. Create `ic_summary` containing:
   - mean IC;
   - standard deviation of IC;
   - annualized IC information ratio;
   - fraction of positive IC observations.

### Important

Do not accidentally correlate the signal with same-day returns.


In [ ]:
# TODO: create daily_ic and ic_summary



In [ ]:
# Validation
assert isinstance(daily_ic, pd.Series)
assert {"mean_ic", "std_ic", "ic_ir", "positive_fraction"} <= set(ic_summary.index)
assert 0 <= ic_summary["positive_fraction"] <= 1
print("Exercise 12 passed.")

## 13. Quantile portfolio test

Perform a cross-sectional signal-sorting test.

### Tasks

On every date:

1. Rank assets by `cs_signal`.
2. Assign them to four quantiles.
3. Compute next-day equal-weight return for each quantile.
4. Store returns in `quantile_returns` with columns `Q1`, `Q2`, `Q3`, `Q4`.
5. Create the long-short spread `Q4_minus_Q1`.

### Notes

- Handle dates with insufficient valid assets.
- The signal must precede the return.
- This is a research diagnostic, separate from the production portfolio.


In [ ]:
# TODO: create quantile_returns and Q4_minus_Q1



In [ ]:
# Validation
assert list(quantile_returns.columns) == ["Q1", "Q2", "Q3", "Q4"]
assert isinstance(Q4_minus_Q1, pd.Series)
assert Q4_minus_Q1.index.equals(quantile_returns.index)
print("Exercise 13 passed.")

## 14. Regime analysis

Define two market regimes using the 60-day realized volatility of the equally weighted equity basket:

- `High vol`: above its expanding historical median;
- `Low vol`: at or below its expanding historical median.

### Tasks

Create:

- `equity_basket_return`;
- `equity_realized_vol`;
- `regime`;
- `regime_stats`, containing annualized return, volatility, Sharpe ratio, and observation count for `targeted_returns` in each regime.

### Constraint

The regime classification used on date `t` must not use information from date `t`.


In [ ]:
# TODO: create regime variables and regime_stats



In [ ]:
# Validation
assert set(regime.dropna().unique()) <= {"High vol", "Low vol"}
assert {"ann_return", "ann_vol", "sharpe", "n_obs"} <= set(regime_stats.columns)
print("Exercise 14 passed.")

## 15. Parameter robustness

Evaluate momentum lookback horizons of:

- 63 days;
- 126 days;
- 252 days.

Use a fixed 21-day skip for every horizon.

For each lookback:

1. construct a cross-sectional momentum score;
2. volatility-scale it using `inv_vol`;
3. impose asset-class neutrality;
4. normalize gross exposure to 1;
5. lag positions by one day;
6. subtract the same transaction-cost model;
7. compute annualized return, volatility, Sharpe ratio, maximum drawdown, and average daily turnover.

Store the results in `robustness`.

### This exercise may use a loop

The main objective is to keep the implementation readable and avoid duplicated logic.


In [ ]:
# TODO: create robustness



In [ ]:
# Validation
assert robustness.index.equals(pd.Index([63, 126, 252], name="lookback"))
assert {"ann_return", "ann_vol", "sharpe", "max_drawdown", "avg_daily_turnover"} <= set(robustness.columns)
print("Exercise 15 passed.")

## 16. Walk-forward evaluation

Split the backtest into:

- training period: 2012–2018;
- validation period: 2019–2021;
- test period: 2022–2025.

### Tasks

Using the `robustness` logic:

1. Select the lookback with the highest Sharpe ratio during the training period.
2. Report its validation-period statistics.
3. Do not change the parameter after observing validation results.
4. Report final test-period statistics.
5. Store the selected parameter in `selected_lookback`.
6. Store results in `walk_forward_summary`.

### Objective

Demonstrate the distinction between:

- in-sample parameter selection;
- validation;
- untouched out-of-sample testing.


In [ ]:
# TODO: perform the walk-forward analysis



In [ ]:
# Validation
assert selected_lookback in {63, 126, 252}
assert {"Train", "Validation", "Test"} <= set(walk_forward_summary.index)
assert {"ann_return", "ann_vol", "sharpe", "max_drawdown"} <= set(walk_forward_summary.columns)
print("Exercise 16 passed.")

## 17. Visualization

Produce four separate figures:

1. Gross, net, and volatility-targeted equity curves.
2. Underwater drawdown chart for the targeted strategy.
3. Rolling 252-day Sharpe ratio of the targeted strategy.
4. Cumulative performance of the four signal quantiles.

### Requirements

- Use clear titles and axis labels.
- Include legends where useful.
- Do not use subplots.
- Do not hard-code the plotted results.


In [ ]:
# TODO: create the four figures



## 18. Research memo

Write a concise research conclusion in Markdown.

Address:

1. Is the signal economically meaningful?
2. How much performance is lost to transaction costs?
3. Does volatility targeting improve the return distribution?
4. Is performance stable across momentum horizons?
5. Does the strategy behave differently in high- and low-volatility regimes?
6. What evidence of overfitting remains?
7. What would you test before deploying real capital?

Do not only quote statistics. Interpret them.


### TODO: write your research memo here



# Optional advanced extensions

These are intentionally open-ended.

## A. Nonlinear market-impact model

Replace linear costs with:

\[
\text{cost}_{i,t}
=
c_i |\Delta w_{i,t}|
+
\lambda
\left(
\frac{|\Delta w_{i,t}| \times \text{AUM}}
{\text{ADV}_{i,t}}
\right)^{3/2}
\]

Study strategy capacity across multiple AUM levels.

## B. Covariance-aware portfolio construction

Replace inverse-volatility scaling with a covariance-based optimizer:

\[
\max_w \quad s_t^	op w - \frac{\gamma}{2}w^	op\Sigma_t w
\]

subject to:

- gross exposure limit;
- asset-class neutrality;
- per-asset position limits.

## C. Signal combination

Combine:

- 12-1 momentum;
- 6-1 momentum;
- short-term reversal;
- exponentially weighted trend.

Evaluate whether combination improves out-of-sample Sharpe.

## D. Bootstrap inference

Use block bootstrap methods to estimate confidence intervals for:

- Sharpe ratio;
- maximum drawdown;
- mean information coefficient.

## E. Deflated Sharpe ratio

Estimate whether the best-performing parameter remains statistically convincing after accounting for multiple testing.
